In [ ]:
import pickle
import sys
sys.path.append("../")
sys.path.append("../../")

from hparams import HParams
from hps import Hyperparams
from vae import VAE

from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.manifold import TSNE
import seaborn as sns
import nltk
from nltk.corpus import stopwords
import string
import numpy as np
import os
import glob
from os.path import join as opj
import h5py  
import matplotlib.pyplot as plt
import pandas as pd
import nibabel as nib
from scipy.io import loadmat
import torch
from torch.utils.data import Dataset, Subset, DataLoader
import json
from PIL import Image
# from diffusers import VersatileDiffusionPipeline
# from diffusers import VersatileDiffusionDualGuidedPipeline
from diffusers.models import AutoencoderKL, DualTransformer2DModel, Transformer2DModel, UNet2DConditionModel
from versatile_diffusion_dual_guided import VersatileDiffusionDualGuidedPipeline
from versatile_diffusion_dual_guided_fake_images import VersatileDiffusionDualGuidedFromCLIPEmbeddingPipeline
from autoencoder import *
from torchsummary import summary
import torchvision
import tqdm
from sklearn.linear_model import Ridge
import pickle
import wandb
from diffusers.utils import (
    PIL_INTERPOLATION,
    deprecate,
    is_accelerate_available,
    is_accelerate_version,
    logging,
    randn_tensor,
    replace_example_docstring,
)

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

from encoding_models import *
from decoding import *
import nilearn
from scipy import stats
from nilearn import plotting
import matplotlib.patches as patches
from torch.nn import functional as F

## Goal: Decode figures for visualization

In [ ]:
base_path="/home/matteo/brain-diffuser/data"
timeseries_path=opj(base_path,"nsddata_timeseries")
betas_path=opj(base_path,"nsddata_betas")

stimuli_path=opj(base_path,"nsddata_stimuli","stimuli","nsd")
stim_file_path=opj(stimuli_path,"nsd_stimuli.hdf5")
sub="subj01"
mod="func1pt8mm"
subj_data_path=opj(timeseries_path,"ppdata",sub,mod,"timeseries")
subj_betas_path=opj(betas_path,"ppdata",sub,mod,"betas_assumehrf")

subj_betas_roi_extracted_path=opj(base_path,"processed_roi",sub,mod)

stim_order_path=opj(base_path,"nsddata","experiments","nsd","nsd_expdesign.mat")
stim_info_path=opj(base_path,"nsddata","experiments","nsd","nsd_stim_info_merged.csv")
stim_captions_train_path=opj(base_path,"nsddata_stimuli","stimuli","nsd","annotations",f"captions_train2017.json")
stim_captions_val_path=opj(base_path,"nsddata_stimuli","stimuli","nsd","annotations",f"captions_val2017.json")

processed_data=opj(base_path,"processed_data",sub)
sub_idx=int(sub.split("0")[-1])


fmri_test_data=opj(processed_data,f"nsd_test_fmriavg_nsdgeneral_sub{sub_idx}.npy")
fmri_test_data=np.load(fmri_test_data)

# load normalizing values 
mean=torch.load(f"../../models/{sub}/train_fmri_mean.pt").numpy()
std=torch.load(f"../../models/{sub}/train_fmri_std.pt").numpy()


fmri_test_data= (fmri_test_data-mean)/std
# fmri_test_data=np.nan_to_num(fmri_test_data)
imgs_test_data=opj(processed_data,f"nsd_test_stim_sub{sub_idx}.npy")

imgs_test_data=np.load(imgs_test_data).astype(np.uint8)




In [ ]:
device="cuda:0"

In [ ]:
subj_path=opj("/home/matteo/explore_NSD","models",sub)
brain_decoder=BrainDiffuserPretrainedDecoder(subj_path=subj_path,device=device)

In [ ]:
modality="fmri"
N=30

## Remove useful indices 
indices=[25,31,68,121,126,318,384,492,531,606,702,860]

indices2=[70,116,165,261,278,363,451,774]
indices3=[41,205,230,411,428,446,502,777]
extra=[95,905]
comparison_indices=indices+indices2+indices3+extra

indices = comparison_indices


fmri_comparison_data =fmri_test_data[indices]



imgs_comparison_data = imgs_test_data[indices]

In [ ]:
target_thr = 25
# concepts = ["man","woman","indoor scene","outdoor scene","day","night","summer","winter","sad","happy","cat","dog","crowded","empty"]

concepts = ["man","woman","indoor scene","outdoor scene","day","night","summer","winter","sad","happy","crowded","empty"]



pert_dir = opj("models_noseed",sub,"encoding",f"algebra_l2_SINGLE_thr_{target_thr}", "fmri")

In [ ]:
target_idx = 1
scaling = 2

tgt_dir=f"results_star/{sub}/{scaling}"
os.makedirs(tgt_dir, exist_ok=True)


In [ ]:


apply_norm = True

decoded_images= []
for target_idx in range(len(imgs_test_data)):
    print(target_idx,"/30")
    base_pattern = torch.tensor(fmri_comparison_data[target_idx]).unsqueeze(0)

    perturbations = []
    algebric= [base_pattern]



    for concept in concepts:
        perturbation =  torch.load(opj(pert_dir,f"{concept}_thr_{target_thr}",f"diff_{target_idx}.pt")).unsqueeze(0)
        perturbations.append(perturbation)

        fmri_algebra = base_pattern + scaling*perturbation

        if apply_norm:
            norm = fmri_algebra.norm(1)

            normed = F.normalize(fmri_algebra,dim=1)
            fmri_algebra=norm*normed/normed.norm(1)

        algebric.append(fmri_algebra)

    algebric = torch.cat(algebric,0)
    algebra_imgs,_=brain_decoder.decode(algebric,num_inference_steps=37)
    
    decoded_images.append(algebra_imgs)

In [ ]:
algebric.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import to_rgb



for tgt_idx in range(30):

# tgt_idx = 1

    img_center = decoded_images[tgt_idx][0]
    img_endpoints =  decoded_images[tgt_idx][1:]


    # Create a figure and axis with equal aspect ratio
    fig, ax = plt.subplots(figsize=(24,24))
    ax.set_aspect('equal')

    # Define angles for the arrows and images (12 equally spaced angles)
    theta = np.linspace(0, 2 * np.pi, num=12, endpoint=False)

    # Parameters for arrows
    arrow_length = 1.2
    arrow_dx = arrow_length * np.cos(theta)
    arrow_dy = arrow_length * np.sin(theta)

    # Draw arrows pointing outward from the central point (before drawing the central image)
    for i in range(12):
        ax.arrow(
            0, 0, arrow_dx[i], arrow_dy[i],
            head_width=0.05, head_length=0.1, fc='k', ec='k', length_includes_head=True,
            zorder=1  # Lower zorder to place arrows behind images
        )


    # Add labels directly on the arrows
    text_distance = 2*arrow_length / 3  # Place text halfway along the arrow

    offset = 0.1 # Offset distance from the arrow
    for i in range(12):
        # Calculate offset position slightly above the arrow direction
        x_label = (text_distance + offset) * np.cos(theta[i])
        y_label = (text_distance + offset) * np.sin(theta[i])

        # Rotate text to align with arrow direction (convert to degrees and adjust for correct orientation)
        angle = np.degrees(theta[i])
        # ax.text(
        #     x_label, y_label, concepts[i],
        #     rotation=angle,
        #     rotation_mode='anchor',  # Keep text aligned with the arrow
        #     ha='center', va='center',
        #     fontsize=12, zorder=4,
        #     bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1)  # Optional: background for better visibility
        # )
        # Rotate text to align with arrow direction (convert to degrees and adjust for correct orientation)
        angle = np.degrees(theta[i])
        if (angle > 180 and angle <= 270) or (angle > 90 and angle <= 180) :
            angle -=180
        ax.text(
            x_label, y_label, concepts[i],
            rotation=angle,
            rotation_mode='anchor',  # Keep text aligned with the arrow
            ha='center', va='center',
            fontsize=18, zorder=6,
            bbox=dict(facecolor='white', alpha=1, edgecolor='none', pad=1)  # Optional: background for better visibility
        )

    # Generate a placeholder central image (replace with your own image as needed)
    ax.imshow(img_center, extent=[-0.5, 0.5, -0.5, 0.5], zorder=3)  # Higher zorder

    # Positions for the outer images
    R_image = 1.5
    x_image = R_image * np.cos(theta)
    y_image = R_image * np.sin(theta)

    # Create placeholder images with different colors
    colors = [
        'red', 'green', 'blue', 'yellow', 'cyan', 'magenta',
        'orange', 'purple', 'brown', 'pink', 'gray', 'olive'
    ]


    # Display the outer images at the end of each arrow
    image_size = 0.5
    for i in range(12):
        x = x_image[i]
        y = y_image[i]
        extent = [
            x - image_size / 2, x + image_size / 2,
            y - image_size / 2, y + image_size / 2
        ]
        ax.imshow(img_endpoints[i], extent=extent, zorder=3)  # Set zorder appropriately

    # Set plot limits and remove axes for a cleaner look
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.axis('off')
    plt.savefig(opj(tgt_dir,f"{tgt_idx}.png"))
    plt.savefig(opj(tgt_dir,f"{tgt_idx}.pdf"),format="pdf")
    
    # Display the figure
    plt.show()


## Brain Figures

I just want to replicate the same figures but with brain patterns

In [ ]:
from nsdcode.nsd_mapdata import NSDmapdata
from nsdcode.nsd_datalocation import nsd_datalocation
from nsdcode.nsd_output import nsd_write_fs
from nsdcode.utils import makeimagestack

In [ ]:
nsd = NSDmapdata("/home/matteo/storage/NSD_transformrs")

In [ ]:
roi_path=opj(base_path,"nsddata","ppdata",sub,mod,"roi")

roi=nib.load(opj(roi_path,"nsdgeneral.nii.gz")).get_fdata()
coordinates=np.array(np.where(roi==1)).T

nsdroi=nib.load(opj(roi_path,"nsdgeneral.nii.gz"))


nsdroi= nilearn.image.math_img("img > 0" , img=nsdroi)
# out_dir=f"models/{sub}/encoding/outputs/roi_ttest_contrast"

roi_path=opj(base_path,"nsddata","ppdata",sub,mod,"roi","nsdgeneral.nii.gz")

# img_mean=mean_img([i for i in glob.glob(opj(f"/home/matteo/data/NSD/nsddata_betas/ppdata/{sub}/{mod}/betas_assumehrf","*.nii.gz")) if "betas" in i][0])
img_mean = nib.load(f"{sub}_mean.nii.gz")
data=img_mean.get_fdata()

roi=nib.load(roi_path).get_fdata()
coordinates=np.array(np.where(roi==1)).T


In [ ]:
target_thr = 25
# concepts = ["man","woman","indoor scene","outdoor scene","day","night","summer","winter","sad","happy","cat","dog","crowded","empty"]

concepts = ["man","woman","indoor scene","outdoor scene","day","night","summer","winter","sad","happy","crowded","empty"]



pert_dir = opj("models_noseed",sub,"encoding",f"algebra_l2_SINGLE_thr_{target_thr}", "fmri")

In [ ]:
pert_dir_vis = opj("models_noseed",sub,"encoding",f"algebra_l2_SINGLE_thr_{90}", "fmri")


In [ ]:
base_pattern = fmri_test_data[0]

base_map=np.zeros(data.shape)

for v,(x,y,z) in enumerate(coordinates):
    base_map[x,y,z]=base_pattern[v]

base_map=nib.Nifti1Image(base_map,affine=img_mean.affine)
base_map=nilearn.image.threshold_img(base_map,threshold=0,cluster_threshold=0,mask_img=nsdroi)

pos_mni=nsd.fit(
    sub_idx,
    'func1pt8',
    'MNI',
    base_map.get_fdata(),
    'nearest',
    badval=0,
    outputfile="base_concept.nii.gz")


display = nilearn.plotting.plot_glass_brain("base_concept.nii.gz")
display.savefig(opj(tgt_dir, f"base_pattern.png"))


In [ ]:
base_MNI=nib.load("base_concept.nii.gz")

for concept in tqdm.tqdm(concepts):
    pert = torch.load(opj(pert_dir_vis,f"{concept}_thr_{90}",f"diff_0.pt")) ## pert are same for all base patterns
    thresholded_map=np.zeros(data.shape)

    for v,(x,y,z) in enumerate(coordinates):
        thresholded_map[x,y,z]=pert[v]

    thresholded_map=nib.Nifti1Image(thresholded_map,affine=img_mean.affine)
    thresholded_map=nilearn.image.threshold_img(thresholded_map,threshold=0,cluster_threshold=20,mask_img=nsdroi)

    pos_mni=nsd.fit(
        sub_idx,
        'func1pt8',
        'MNI',
        thresholded_map.get_fdata(),
        'nearest',
        badval=0,
        outputfile=opj(tgt_dir,f"{concept}_thr_{90}.nii.gz")) 

    added = nib.load(opj(tgt_dir,f"{concept}_thr_{90}.nii.gz")) 

    final_sum = nilearn.image.math_img("img1+4*img2",img1=base_MNI,img2=added)

    nib.save(final_sum,opj(tgt_dir,f"FINAL_{concept}_thr_{90}.nii.gz"))


    display = nilearn.plotting.plot_glass_brain(opj(tgt_dir,f"{concept}_thr_{90}.nii.gz"),title=concept)
    display.savefig(opj(tgt_dir, f"pertubation_img_{concept}.png"))

    display.close()

    display = nilearn.plotting.plot_glass_brain(opj(tgt_dir,f"FINAL_{concept}_thr_{90}.nii.gz"),title=concept)
    display.savefig(opj(tgt_dir, f"FINAL_img_{concept}.png"))

    display.close()

    
